# Document RAG Assistant
## Naive RAG Pipeline — Lecture 1 — Hugging Face Edition

This notebook builds RAG from raw Python without LangChain or LlamaIndex.

### Architecture

**PDF/TXT → Clean → Chunk → Local Embeddings → ChromaDB → Retrieve → Hugging Face LLM → Answer**

- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`, executed locally
- Vector database: ChromaDB
- LLM: configurable Hugging Face hosted instruction model
- Authentication: Hugging Face User Access Token

Local embeddings avoid a paid embedding API. Hugging Face hosted inference can have free-tier limits, so this notebook is designed for learning and portfolio demos rather than unlimited free production usage.


## Step 1 — Setup & Installations

We use direct Python libraries only. `PyPDF2` reads PDFs, `sentence-transformers` creates local embeddings, `chromadb` stores vectors, `huggingface_hub` calls the hosted LLM, and `numpy` helps inspect vectors.

The embedding model is downloaded the first time and then cached locally.


In [ ]:
%pip install -q PyPDF2 huggingface_hub sentence-transformers chromadb numpy


### Configure your Hugging Face token

Store the token in an environment variable named `HF_TOKEN`. Never commit a real token to GitHub.

For a temporary local notebook test, uncomment the example assignment in the next cell, paste your token, run the notebook, and remove the token before publishing.


In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import chromadb
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

# Recommended: set HF_TOKEN outside the notebook.
# Temporary local test only:
# os.environ["HF_TOKEN"] = "hf_your_token_here"
# NEVER push a real token to GitHub.

hf_token = os.environ.get("HF_TOKEN")

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model: {LLM_MODEL}")

try:
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    print("✅ Local embedding model loaded.")
except Exception as error:
    embedding_model = None
    print(f"❌ Embedding model error: {error}")

if hf_token:
    hf_client = InferenceClient(
        provider="auto",
        api_key=hf_token,
        timeout=120
    )
    print("✅ Hugging Face InferenceClient is ready.")
else:
    hf_client = None
    print("⚠️ HF_TOKEN is not set.")
    print("Local embeddings still work, but hosted LLM generation needs a token.")

print("✅ Libraries imported successfully.")


## Step 2 — Document Loading

RAG begins with source documents. Instead of storing plain strings only, we attach **metadata** such as filename and page number so retrieved answers can later show where information came from.

The notebook looks for `sample_document.pdf` and `company_notes.txt`. If either file is missing, beginner-friendly inline fallback content is used so the loading and chunking lessons can still run.


In [ ]:
PDF_PATH = "sample_document.pdf"
TXT_PATH = "company_notes.txt"

# A TXT sample is created automatically if it does not exist.
sample_txt = """
NovaCare Analytics is a fictional healthcare technology company founded in 2021.
Its main product is InsightFlow, a platform that helps hospitals analyze operational data.
InsightFlow can process appointment trends, equipment usage, and patient flow statistics.

The company uses Python for data processing and PostgreSQL for structured data storage.
Its machine learning team experiments with forecasting models to predict appointment demand.
NovaCare's internal policy requires human review before an AI-generated recommendation is used for a critical decision.

NovaCare has offices in Hyderabad and Dubai. The Dubai team focuses on regional partnerships,
while the Hyderabad engineering team develops data and machine learning services.
"""

if not Path(TXT_PATH).exists():
    Path(TXT_PATH).write_text(sample_txt.strip(), encoding="utf-8")
    print(f"📝 Created sample TXT file: {TXT_PATH}")

documents = []

def load_pdf(file_path):
    """Load a PDF page by page and return document dictionaries with metadata."""
    loaded_docs = []
    try:
        reader = PdfReader(file_path)

        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""

            loaded_docs.append({
                "content": page_text,
                "metadata": {
                    "source": Path(file_path).name,
                    "page": page_number
                }
            })

        print(f"✅ Loaded PDF: {file_path}")
    except Exception as error:
        print(f"⚠️ Could not load PDF: {error}")

    return loaded_docs


def load_txt(file_path):
    """Load a TXT file and return one document dictionary with source metadata."""
    loaded_docs = []
    try:
        with open(file_path, "r", encoding="utf-8", errors="replace") as file:
            text = file.read()

        loaded_docs.append({
            "content": text,
            "metadata": {
                "source": Path(file_path).name,
                "page": 1
            }
        })

        print(f"✅ Loaded TXT: {file_path}")
    except Exception as error:
        print(f"⚠️ Could not load TXT: {error}")

    return loaded_docs


# Load the real PDF if present.
if Path(PDF_PATH).exists():
    documents.extend(load_pdf(PDF_PATH))
else:
    print(f"⚠️ {PDF_PATH} was not found.")
    print("Using inline PDF fallback content. Replace it with your own PDF for a real extraction demo.")

    pdf_fallback_pages = [
        """
        Retrieval-Augmented Generation, commonly called RAG, combines information retrieval
        with a large language model. The retriever searches a knowledge source for relevant
        text. The language model receives the retrieved text as context before answering.
        This approach can reduce unsupported answers because the model is instructed to use
        supplied evidence.
        """,
        """
        Text chunking divides long documents into smaller pieces. Very large chunks may contain
        unrelated information, while very small chunks may lose important context. Overlap can
        preserve information near chunk boundaries. Recursive chunking tries meaningful separators
        such as paragraphs and sentences before falling back to character limits.
        """,
        """
        Embeddings are numerical vectors that represent semantic meaning. Texts with related meaning
        tend to have vectors that are closer in vector space. A vector database stores embeddings
        and supports similarity search. Retrieval quality depends on document quality, chunking,
        embedding choice, and the number of results retrieved.
        """
    ]

    for page_number, page_text in enumerate(pdf_fallback_pages, start=1):
        documents.append({
            "content": page_text,
            "metadata": {
                "source": "sample_document.pdf_fallback",
                "page": page_number
            }
        })

documents.extend(load_txt(TXT_PATH))

print("\n" + "=" * 70)
for index, document in enumerate(documents):
    print(f"\nDOCUMENT {index + 1}")
    print("Metadata:", document["metadata"])
    print("First 500 characters:")
    print(document["content"][:500])

print(f"\n📚 Total number of documents loaded: {len(documents)}")


## Step 3 — Text Preprocessing

Extracted text often contains repeated whitespace, strange Unicode characters, or encoding artifacts. Cleaning makes the text more consistent before chunking and embedding.

Lowercasing is optional. It can normalize text, but it may remove useful distinctions in names or acronyms, so this project keeps the original letter case.


In [ ]:
def clean_text(text, lowercase=False):
    """Clean extracted text while preserving readable punctuation and optional letter case."""
    # Normalize Unicode representation.
    text = unicodedata.normalize("NFKC", text)

    # Replace common encoding artifacts.
    text = text.replace("\x00", " ")
    text = text.replace("\ufffd", " ")

    # Keep letters, numbers, common punctuation, and whitespace.
    text = re.sub(r"[^\w\s.,!?;:'\"()\-/%]", " ", text, flags=re.UNICODE)

    # Collapse repeated spaces, tabs, and newlines.
    text = re.sub(r"\s+", " ", text).strip()

    if lowercase:
        text = text.lower()

    return text


before_cleaning = documents[0]["content"]

cleaned_documents = []

for document in documents:
    cleaned_documents.append({
        "content": clean_text(document["content"], lowercase=False),
        "metadata": document["metadata"].copy()
    })

documents = cleaned_documents

print("BEFORE CLEANING:")
print(before_cleaning[:500])

print("\nAFTER CLEANING:")
print(documents[0]["content"][:500])

print(f"\n✅ Cleaned {len(documents)} documents while keeping metadata attached.")


## Step 4 — Chunking Strategy 1: Fixed-Size Characters

LLMs and embedding systems work better when long documents are divided into smaller pieces called **chunks**. The simplest method cuts text every fixed number of characters.

This is easy to understand, but it can cut a sentence or idea in the middle.


In [ ]:
def fixed_size_chunk(text, chunk_size=500):
    """Split text into consecutive fixed-size character chunks."""
    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), chunk_size)
    ]


fixed_chunks = fixed_size_chunk(documents[0]["content"], chunk_size=500)

print(f"Number of fixed-size chunks: {len(fixed_chunks)}")
print("\nFIRST FIXED-SIZE CHUNK:")
print(fixed_chunks[0])
print(f"\nFirst chunk length: {len(fixed_chunks[0])} characters")


## Step 4 — Chunking Strategy 2: Sliding Window with Overlap

A sliding window repeats part of the previous chunk in the next chunk. The overlap helps preserve ideas that happen near a chunk boundary.

The trade-off is extra storage and repeated text because some characters appear in multiple chunks.


In [ ]:
def sliding_window_chunk(text, chunk_size=500, overlap=100):
    """Split text into overlapping character windows."""
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    step_size = chunk_size - overlap

    for start in range(0, len(text), step_size):
        chunk = text[start:start + chunk_size]

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(text):
            break

    return chunks


sliding_chunks = sliding_window_chunk(
    documents[0]["content"],
    chunk_size=500,
    overlap=100
)

print(f"Number of sliding-window chunks: {len(sliding_chunks)}")

if len(sliding_chunks) >= 2:
    print("\nLAST 100 CHARACTERS OF CHUNK 1:")
    print(sliding_chunks[0][-100:])

    print("\nFIRST 100 CHARACTERS OF CHUNK 2:")
    print(sliding_chunks[1][:100])

    print("\nDo they match?")
    print(sliding_chunks[0][-100:] == sliding_chunks[1][:100])
else:
    print("The sample document is too short to demonstrate two overlapping chunks.")


## Step 4 — Chunking Strategy 3: Recursive Chunking

Recursive chunking tries to preserve meaning by using natural boundaries before cutting by character count. We first prefer paragraphs, then sentences, and only use raw character limits when a single sentence is still too large.

This is usually more meaningful than blindly cutting every 500 characters because related sentences are more likely to stay together. We will use recursive chunking for the rest of this project.


In [ ]:
def recursive_chunk(text, chunk_size=500):
    """Split text using paragraphs, then sentences, then character limits."""
    def split_large_piece(piece):
        """Split an oversized piece by sentences or finally by characters."""
        piece = piece.strip()

        if not piece:
            return []

        if len(piece) <= chunk_size:
            return [piece]

        # Try sentence boundaries.
        sentences = re.split(r"(?<=[.!?])\s+", piece)

        if len(sentences) > 1:
            sentence_chunks = []
            current_chunk = ""

            for sentence in sentences:
                candidate = f"{current_chunk} {sentence}".strip()

                if len(candidate) <= chunk_size:
                    current_chunk = candidate
                else:
                    if current_chunk:
                        sentence_chunks.append(current_chunk)

                    if len(sentence) <= chunk_size:
                        current_chunk = sentence
                    else:
                        # Final fallback: raw character slicing.
                        sentence_chunks.extend(
                            sentence[start:start + chunk_size]
                            for start in range(0, len(sentence), chunk_size)
                        )
                        current_chunk = ""

            if current_chunk:
                sentence_chunks.append(current_chunk)

            return sentence_chunks

        # No useful sentence boundary exists.
        return [
            piece[start:start + chunk_size]
            for start in range(0, len(piece), chunk_size)
        ]

    # Paragraphs are the first and most meaningful separator.
    paragraphs = re.split(r"\n\s*\n", text)

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        candidate = f"{current_chunk}\n\n{paragraph}".strip()

        if len(candidate) <= chunk_size:
            current_chunk = candidate
        else:
            if current_chunk:
                chunks.append(current_chunk)

            paragraph_parts = split_large_piece(paragraph)

            if paragraph_parts:
                chunks.extend(paragraph_parts[:-1])
                current_chunk = paragraph_parts[-1]
            else:
                current_chunk = ""

    if current_chunk:
        chunks.append(current_chunk)

    return chunks


recursive_demo_chunks = recursive_chunk(
    documents[0]["content"],
    chunk_size=500
)

print(f"Number of recursive chunks: {len(recursive_demo_chunks)}")

for index, chunk in enumerate(recursive_demo_chunks[:3], start=1):
    print(f"\n--- RECURSIVE CHUNK {index} ---")
    print(chunk)


### Create chunk dictionaries with metadata

A useful RAG chunk contains both text and metadata. The text is embedded and retrieved, while metadata lets us trace a result back to its source and page.

Each chunk receives a `chunk_id` so it can also have a unique ID in ChromaDB.


In [ ]:
chunks = []
global_chunk_id = 0

for document in documents:
    document_chunks = recursive_chunk(
        document["content"],
        chunk_size=500
    )

    for chunk_text in document_chunks:
        chunks.append({
            "content": chunk_text,
            "metadata": {
                "source": document["metadata"]["source"],
                "page": document["metadata"]["page"],
                "chunk_id": global_chunk_id
            }
        })

        global_chunk_id += 1

print(f"📦 Total chunks created: {len(chunks)}")

print("\nSAMPLE CHUNK:")
print(chunks[0]["content"])

print("\nSAMPLE METADATA:")
print(chunks[0]["metadata"])


## Step 5 — Embeddings: Turning Text into Numbers That Capture Meaning

An embedding converts text into a numerical vector representing semantic meaning. Related texts tend to be closer in vector space, which enables semantic search.

We use `sentence-transformers/all-MiniLM-L6-v2` locally. Both document chunks and user queries must use the same embedding model.


In [ ]:
def get_embeddings(texts, batch_size=32):
    """Generate local Sentence Transformer embeddings for a list of texts."""
    if embedding_model is None:
        print("❌ Embedding model is unavailable.")
        return []

    try:
        embeddings = embedding_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        embedding_lists = embeddings.tolist()

        print(f"✅ Generated {len(embedding_lists)} local embeddings.")

        return embedding_lists

    except Exception as error:
        print(f"❌ Embedding error: {error}")
        return []


chunk_texts = [chunk["content"] for chunk in chunks]
chunk_embeddings = get_embeddings(chunk_texts)

sample_query = "How does retrieval augmented generation work?"
sample_query_embedding = get_embeddings([sample_query])

if chunk_embeddings:
    embedding_length = len(chunk_embeddings[0])

    print(f"\nShape of one embedding vector: ({embedding_length},)")
    print(f"Each chunk is now a vector of {embedding_length} dimensions.")

    print("\nFirst 10 numbers of the first vector:")
    print(np.array(chunk_embeddings[0][:10]))

if sample_query_embedding:
    print(
        "\nSample query embedding length:",
        len(sample_query_embedding[0])
    )
    print("✅ Documents and queries use the same embedding process.")


## Step 6 — Vector Store with ChromaDB: A Search Engine for Meaning, Not Keywords

A vector database stores embeddings and searches for vectors that are close to a query vector. Unlike simple keyword matching, vector search is designed to find semantically related text.

We use an in-memory ChromaDB client for this lecture. Restarting the notebook clears the collection, which is convenient for learning and experimentation.


In [ ]:
chroma_client = chromadb.Client()

# Delete an old collection from a previous notebook run.
try:
    chroma_client.delete_collection(name="document_rag")
    print("🧹 Removed previous document_rag collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="document_rag"
)

if chunk_embeddings and len(chunk_embeddings) == len(chunks):
    collection.add(
        documents=[chunk["content"] for chunk in chunks],
        embeddings=chunk_embeddings,
        metadatas=[chunk["metadata"] for chunk in chunks],
        ids=[
            f"chunk_{chunk['metadata']['chunk_id']}"
            for chunk in chunks
        ]
    )

    print(f"✅ Stored {len(chunks)} chunks in ChromaDB")
    print(f"Collection count: {collection.count()}")
else:
    print("⚠️ Chunks were not stored because embeddings are unavailable.")
    print("Set OPENAI_API_KEY, rerun Step 5, and then rerun this cell.")


## Step 7 — Retrieval: Finding the Closest Meaning Match

Cosine-style vector similarity asks whether two vectors point in a similar semantic direction. In simple terms, the query vector is compared with stored chunk vectors to find chunks with the closest meaning.

ChromaDB returns distances with the retrieved results. For the default distance behavior, smaller distance values indicate closer matches.


In [ ]:
def retrieve(query, n_results=3):
    """Embed a query and retrieve relevant chunk text, distances, and metadata."""
    query_embeddings = get_embeddings([query])

    if not query_embeddings:
        return {
            "documents": [],
            "distances": [],
            "metadatas": []
        }

    try:
        result = collection.query(
            query_embeddings=query_embeddings,
            n_results=min(n_results, collection.count()),
            include=["documents", "distances", "metadatas"]
        )

        return {
            "documents": result["documents"][0],
            "distances": result["distances"][0],
            "metadatas": result["metadatas"][0]
        }

    except Exception as error:
        print(f"❌ ChromaDB retrieval error: {error}")

        return {
            "documents": [],
            "distances": [],
            "metadatas": []
        }


sample_queries = [
    "What is RAG and how does it answer questions?",
    "What technology does NovaCare use for structured data?",
    "Why is recursive chunking useful?"
]

for query in sample_queries:
    print("\n" + "=" * 80)
    print(f"QUERY: {query}")

    results = retrieve(query, n_results=3)

    for index, (
        document,
        distance,
        metadata
    ) in enumerate(
        zip(
            results["documents"],
            results["distances"],
            results["metadatas"]
        ),
        start=1
    ):
        print(f"\nRETRIEVED CHUNK {index}")
        print(f"Distance: {distance:.4f}")
        print(f"Source metadata: {metadata}")
        print(document)


## Step 8 — LLM + Prompt Engineering: The RAG Prompt Pattern

Retrieval finds potentially useful evidence, but the LLM still needs clear instructions. A grounded RAG prompt tells the model to answer only from retrieved context and to admit when the context is insufficient.

We separate the **system instruction** from the **user prompt**. The user prompt contains numbered context chunks followed by the question.


In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer questions ONLY based on the "
    "provided context. If the answer is not in the context, say "
    "\"I don't have enough information to answer this.\""
)


def build_prompt(query, retrieved_chunks):
    """Build the grounded user prompt from a query and retrieved chunk texts."""
    numbered_context = "\n\n".join(
        f"[{index}] {chunk_text}"
        for index, chunk_text in enumerate(
            retrieved_chunks,
            start=1
        )
    )

    user_prompt = f"""Context:
{numbered_context}

Question: {query}

Answer based only on the context above. If the context doesn't contain the answer, say "I don't have enough information to answer this."
"""

    return user_prompt


demo_query = "Why can recursive chunking preserve meaning better?"
demo_results = retrieve(demo_query, n_results=3)

demo_prompt = build_prompt(
    demo_query,
    demo_results["documents"]
)

print("SYSTEM MESSAGE:")
print(SYSTEM_PROMPT)

print("\nFULL CONSTRUCTED USER PROMPT:")
print(demo_prompt)


### Send the grounded prompt to a Hugging Face instruction model

The retrieved context is sent to a hosted instruction model through Hugging Face `InferenceClient`. A low temperature of `0.2` reduces randomness for factual RAG answers.

Hosted model/provider availability can change, so the model ID is stored in the `LLM_MODEL` variable near the top of the notebook.


In [ ]:
def ask_llm(query, retrieved_chunks):
    """Build a grounded RAG prompt and ask a Hugging Face hosted LLM."""
    if hf_client is None:
        return "Hugging Face client is unavailable. Set HF_TOKEN first."

    user_prompt = build_prompt(query, retrieved_chunks)

    try:
        response = hf_client.chat_completion(
            model=LLM_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0.2,
            max_tokens=300
        )

        return response.choices[0].message.content

    except Exception as error:
        return (
            f"Hugging Face LLM error: {error}\n\n"
            "Provider availability can change. "
            "If necessary, replace LLM_MODEL with another "
            "chat-compatible Hugging Face Inference Providers model."
        )


if demo_results["documents"]:
    demo_answer = ask_llm(
        demo_query,
        demo_results["documents"]
    )

    print("DEMO ANSWER:")
    print(demo_answer)
else:
    print("⚠️ No retrieved chunks are available yet.")


## Step 9 — Complete RAG Pipeline Function

We now connect retrieval and generation into one function. A user asks a question, the query is embedded, ChromaDB retrieves relevant chunks, and Hugging Face instruction model answers from those chunks.

The function also prints the full flow so students can see that RAG is a sequence of understandable steps rather than a hidden framework.


In [ ]:
def rag_pipeline(query, n_results=3):
    """Run retrieval and grounded LLM generation for one user question."""
    print("\n" + "=" * 80)
    print(f"📝 Query: {query}")

    retrieval_result = retrieve(
        query,
        n_results=n_results
    )

    retrieved_chunks = retrieval_result["documents"]
    metadata_list = retrieval_result["metadatas"]

    print(f"🔍 Retrieved {len(retrieved_chunks)} relevant chunks")

    sources = sorted({
        metadata["source"]
        for metadata in metadata_list
    })

    print(f"📄 Sources: {sources}")

    answer = ask_llm(
        query,
        retrieved_chunks
    )

    print(f"🤖 Answer: {answer}")

    return {
        "query": query,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "distances": retrieval_result["distances"],
        "metadatas": metadata_list,
        "sources": sources
    }


# Example:
# result = rag_pipeline("What is RAG?")


## Step 10 — Live Demo: Test the Pipeline

A good RAG demo should test more than easy questions. The five questions below include answerable questions, a cross-source question, a partially answerable question, and an unrelated question designed to test hallucination control.

For each test, we display abbreviated retrieved chunks, the generated answer, and the source documents used.


In [ ]:
demo_questions = [
    # Answerable from the PDF/fallback RAG content.
    "What is RAG and what two major parts does it combine?",

    # Answerable from the TXT source.
    "What is NovaCare's main product and what does it analyze?",

    # Cross-source: requires ideas from the RAG source and company notes.
    "How could retrieval and embeddings be used with NovaCare's internal information?",

    # Partially answerable: the office is known, but employee count is not.
    "Where does NovaCare have offices and how many employees work in each office?",

    # Not present in the documents.
    "Who won the FIFA World Cup in 2018?"
]

demo_outputs = []

for question_number, question in enumerate(
    demo_questions,
    start=1
):
    result = rag_pipeline(
        question,
        n_results=3
    )

    demo_outputs.append(result)

    print(f"\nTEST QUESTION {question_number}: {question}")

    print("\nABBREVIATED RETRIEVED CHUNKS:")

    for chunk_number, chunk_text in enumerate(
        result["retrieved_chunks"],
        start=1
    ):
        abbreviated = (
            chunk_text[:220] + "..."
            if len(chunk_text) > 220
            else chunk_text
        )

        print(
            f"\n[{chunk_number}] {abbreviated}"
        )

    print("\nLLM ANSWER:")
    print(result["answer"])

    print("\nSOURCE DOCUMENTS USED:")
    print(result["sources"])

    print("\n" + "-" * 80)


## Step 11 — Summary & Recap

### Complete RAG pipeline

**Load → Clean → Chunk → Embed → Store → Retrieve → Prompt → Answer**

1. **Load** PDF and TXT documents with metadata.
2. **Clean** extracted text.
3. **Chunk** using fixed-size, sliding-window, and recursive strategies.
4. **Embed** chunks locally with Sentence Transformers.
5. **Store** vectors, text, and metadata in ChromaDB.
6. **Retrieve** semantically relevant chunks for the query.
7. **Prompt** the LLM with numbered context and strict grounding instructions.
8. **Answer** using a Hugging Face hosted instruction model.

### Libraries used

- `PyPDF2` — PDF extraction
- `sentence-transformers` — local embeddings
- `huggingface_hub` — hosted LLM inference
- `chromadb` — vector database
- `numpy` — vector inspection

### Key decisions

| Decision | Value |
|---|---|
| Default chunking | Recursive |
| Chunk size | 500 characters |
| Sliding overlap | 100 characters |
| Embedding model | `all-MiniLM-L6-v2` |
| Embedding execution | Local |
| Retrieval K | 3 |
| LLM | Configurable `LLM_MODEL` |
| Temperature | 0.2 |

### Cost note

The embedding stage runs locally and does not use the Hugging Face/local model. Hugging Face hosted inference may have limited free-tier usage depending on account and provider; it is not unlimited free compute.

### Key lesson

RAG quality depends on document quality, chunking, embeddings, retrieval, prompt grounding, and the LLM. Always inspect retrieved chunks when an answer is weak.
